# 完整 Stage1：旧分配器与分钟增强模型对照

用户指出旧完整策略回撤约 30%，此前分钟卡却显示 42%–48%。本卡修正比较层级：复用 NAS 冻结神经预测及现役动作/恢复模型，统一价格、风险流程、账户初态与费用。固定 5 个分配器/基准、2 档费用、10 个原生账户；0 次拟合、0 次调参。2023–2026 是复用历史诊断，不能成为新的样本外或模拟盘前向业绩。

In [1]:
from pathlib import Path
import sys
import os
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "AGENTS.md").exists() and (p / "src").is_dir())
sys.path.insert(0, str(ROOT))
import json
import qlib
from qlib.data import D
from qlib.workflow import R
import pandas as pd
from IPython.display import display, Markdown
from src.workflow.stage1_minute_comparison import prepare_stage1_minute_comparison, run_stage1_minute_comparison, supplement_stage1_cost_audit
CARD = ROOT / "research/stage1_minute_information_comparison_20261012"
PLAN = json.loads((CARD / "plan.json").read_text())
qlib.init(provider_uri=str(ROOT / "data/qlib/minute_information_pit_allocator_20261012/daily_prices"), region="us", kernels=1)
assert len(D.calendar(start_time=PLAN["signal_start"], end_time=PLAN["as_of"], freq="day")) > 900
PLAN["research_role"]

'correct allocator-only versus complete strategy comparison; zero model fits or parameter search'

## 冻结输入与完整流程

通过 NAS 恢复原 Qlib Recorder，核对输入、预测和模型引用摘要。实际分钟 OHLC 及已公布复权事件仅用于统一历史记账价格；不重算模型特征。源分配器复用现役 EMA/HOLD。开盘动作、强制减仓/修复、风险等待/爬坡与恢复操作均调用当前发布版本的函数。10bp 与 25bp 账户分别从现金开始，分别积累净值峰值和风险状态。

In [2]:
import pickle
CACHE = Path("/tmp/soxl-stage1-minute-comparison-20261012/prepared_inputs.pkl")
if (CARD / "summary.json").exists():
    PREPARED = None
elif CACHE.exists():
    with CACHE.open("rb") as handle:
        PREPARED = pickle.load(handle)
else:
    PREPARED = prepare_stage1_minute_comparison(root=ROOT, card=CARD, plan=PLAN)
    CACHE.parent.mkdir(exist_ok=True)
    with CACHE.open("wb") as handle:
        pickle.dump(PREPARED, handle)
if PREPARED is not None:
    display({k: PREPARED["audit"][k] for k in ["sessions", "first_trade", "last_exit", "new_model_fits"]})

In [3]:
SUMMARY = json.loads((CARD / "summary.json").read_text()) if PREPARED is None else run_stage1_minute_comparison(root=ROOT, card=CARD, plan=PLAN, prepared=PREPARED)
SUMMARY = supplement_stage1_cost_audit(root=ROOT, card=CARD)
assert SUMMARY["model_fits"] == 0 and SUMMARY["native_cases"] == 10
assert SUMMARY["current_release_changed"] is False
assert SUMMARY["forward_qualified_sessions"] == 0
SUMMARY["diagnostic_gates"]

{'net_return_above_soxx_both_costs': False,
 'observed_drawdown_within_30pct_both_costs': False,
 'return_above_refit_and_availability_controls_both_costs': False,
 'original_minute_development_gate_pass': False,
 'independent_forward_gate_pass': False}

In [4]:
with R.uri_context(SUMMARY["recorder_uri"]):
    RECORDER = R.get_recorder(experiment_id=SUMMARY["experiment_id"], recorder_id=SUMMARY["recorder_id"])
    COMPARISON = RECORDER.load_object("analysis/comparison")
    ANNUAL = RECORDER.load_object("analysis/annual")
display(COMPARISON[["cost_bps", "family", "return_pct", "max_drawdown_pct", "cvar5_pct", "wealth_ratio_vs_soxx", "wealth_ratio_vs_base", "open_trade_days", "recovery_action_count", "native_reconciliation_maximum_error"]])
display(ANNUAL.loc[ANNUAL.family.eq("core_path") & ANNUAL.reference.eq("soxx"), ["cost_bps", "year", "return_pct", "benchmark_return_pct", "relative_return_pct", "max_drawdown_pct"]])

,cost_bps,family,return_pct,max_drawdown_pct,cvar5_pct,wealth_ratio_vs_soxx,wealth_ratio_vs_base,open_trade_days,recovery_action_count,native_reconciliation_maximum_error
0,10,frozen_current,219.625810,-26.921785,-4.788869,0.639946,0.808314,799.0,0.0,4.086953e-12
1,10,base,295.422980,-24.427566,-4.497426,0.791705,1.000000,817.0,0.0,5.056178e-12
2,10,availability_control,209.079264,-28.497322,-4.333443,0.618830,0.781642,853.0,0.0,3.184120e-12
3,10,core_path,91.388634,-30.054900,-4.231342,0.383193,0.484010,809.0,0.0,4.103384e-13
4,10,soxx,399.457476,-42.789189,-5.278418,1.000000,1.263097,1.0,0.0,4.440892e-16
5,25,frozen_current,75.080927,-32.379825,-4.413574,0.351067,1.119998,754.0,0.0,3.998801e-12
6,25,base,56.322493,-29.154850,-4.048217,0.313454,1.000000,812.0,0.0,1.631917e-11
7,25,availability_control,156.031750,-29.340539,-4.331991,0.513388,1.637843,854.0,0.0,4.837686e-12
8,25,core_path,98.047708,-30.406628,-4.245188,0.397120,1.266918,805.0,0.0,4.058975e-12
9,25,soxx,398.710158,-42.789189,-5.278418,1.000000,3.190265,1.0,0.0,2.220446e-16


,cost_bps,year,return_pct,benchmark_return_pct,relative_return_pct,max_drawdown_pct
48,10,2023,35.398462,56.666194,-13.575189,-19.900123
49,10,2024,17.198629,20.784920,-2.969154,-18.518924
50,10,2025,-10.116319,47.980662,-39.259846,-19.909893
51,10,2026,34.183473,78.363318,-24.769580,-25.679810
128,25,2023,32.948135,56.431781,-15.012068,-20.418967
129,25,2024,13.970865,20.784920,-5.641478,-18.788494
130,25,2025,-7.099152,47.980662,-37.220954,-18.783307
131,25,2026,40.693506,78.363318,-21.119708,-26.017866


In [5]:
display(Markdown((CARD / "conclusion.md").read_text()))

# 完整 Stage1 对照结论

本次修正比较层级：先前分钟卡评估分配器单层，不能替代旧模拟盘完整策略。复用冻结分配器预测及现役动作/恢复模型，没有训练、调参或切换模拟盘。
固定历史诊断：2023-01-04 至 2026-10-09，944 个完整账户区间。每档费用独立持仓、净值峰值、回撤、风险时钟与 HOLD/恢复动作；Qlib 原生订单、现金、数量逐日核对。

| 费用 | 分配器接完整 Stage1 | 净收益 | 最大回撤 | 相对 SOXX 终值 |
|---|---|---:|---:|---:|
| 10bp | frozen_current | 219.63% | -26.92% | 0.6399 |
| 10bp | base | 295.42% | -24.43% | 0.7917 |
| 10bp | availability_control | 209.08% | -28.50% | 0.6188 |
| 10bp | core_path | 91.39% | -30.05% | 0.3832 |
| 10bp | soxx | 399.46% | -42.79% | 1.0000 |
| 25bp | frozen_current | 75.08% | -32.38% | 0.3511 |
| 25bp | base | 56.32% | -29.15% | 0.3135 |
| 25bp | availability_control | 156.03% | -29.34% | 0.5134 |
| 25bp | core_path | 98.05% | -30.41% | 0.3971 |
| 25bp | soxx | 398.71% | -42.79% | 1.0000 |

旧卡开发期 10bp 最大回撤 −30.08%、25bp −31.88%；旧外层约 −26.92%/−27.32%。这些是已观察的历史结果，不是硬性保证。旧窗口、旧价格来源的数值保留在 legacy_reference.csv，不能直接与扩展窗口相减。

旧卡 25bp 是固定 10bp 开盘/HOLD/恢复事件后重新扣费的压力测试（cost_stress_contract=fixed_target_hold_recovery_events_repriced_at_each_stage_net_nav），没有按 25bp 净值重新决定风控路径。本卡两档费用各自重算状态。两种费用口径必须区分。

年度差异见 regime_metrics.csv。旧单模型与三 seed 重训基线的差异不能全部归因于分钟数据；core_path 必须同时与 base 和 availability_control 对照。
分钟单层的历史改进需要经过完整控制器重新验证。两档费用下候选与对照的排序不同，不能据此重新选模。控制器成本敏感性见 cost_path_differences.csv，净值回撤与风险状态核对见 controller_state_diagnostics.csv。

历史执行价格统一为 NAS 真实分钟及已公布复权事件构建的开盘/首小时结束标记。开盘及 10:30 close 是历史记账标记，不声明真实 09:31/10:31 成交、首次数据接收时点或现金分红资格。模型分钟输入仍为此前冻结的 raw NONE，未用本次复权系数重算特征。
诊断门：{"net_return_above_soxx_both_costs": false, "observed_drawdown_within_30pct_both_costs": false, "return_above_refit_and_availability_controls_both_costs": false, "original_minute_development_gate_pass": false, "independent_forward_gate_pass": false}。
晋升：not_promoted_full_stage1_comparison_diagnostic；原分钟开发门已失败，独立合格前向为 0，objective_achieved=false。比较修正不能重新挑选候选或把复用历史变成新的样本外证据。
下一步：结束此固定对照；保留失败路线。模型优化需要新的有效信息或预先冻结的独立前向对照，不能靠增加同一历史上的参数搜索。
Recorder：9d671fcfb1aa467bb8a9fd83971fb015。行情、大账户及模型引用留 NAS；Git 保留 notebook、瘦卡和代码。


# 完整 Stage1 对照结论

本次修正比较层级：先前分钟卡评估分配器单层，不能替代旧模拟盘完整策略。复用冻结分配器预测及现役动作/恢复模型，没有训练、调参或切换模拟盘。
固定历史诊断：2023-01-04 至 2026-10-09，944 个完整账户区间。每档费用独立持仓、净值峰值、回撤、风险时钟与 HOLD/恢复动作；Qlib 原生订单、现金、数量逐日核对。

| 费用 | 分配器接完整 Stage1 | 净收益 | 最大回撤 | 相对 SOXX 终值 |
|---|---|---:|---:|---:|
| 10bp | frozen_current | 219.63% | -26.92% | 0.6399 |
| 10bp | base | 295.42% | -24.43% | 0.7917 |
| 10bp | availability_control | 209.08% | -28.50% | 0.6188 |
| 10bp | core_path | 91.39% | -30.05% | 0.3832 |
| 10bp | soxx | 399.46% | -42.79% | 1.0000 |
| 25bp | frozen_current | 75.08% | -32.38% | 0.3511 |
| 25bp | base | 56.32% | -29.15% | 0.3135 |
| 25bp | availability_control | 156.03% | -29.34% | 0.5134 |
| 25bp | core_path | 98.05% | -30.41% | 0.3971 |
| 25bp | soxx | 398.71% | -42.79% | 1.0000 |

旧卡开发期 10bp 最大回撤 −30.08%、25bp −31.88%；旧外层约 −26.92%/−27.32%。这些是已观察的历史结果，不是硬性保证。旧窗口、旧价格来源的数值保留在 legacy_reference.csv，不能直接与扩展窗口相减。

旧卡 25bp 是固定 10bp 开盘/HOLD/恢复事件后重新扣费的压力测试（cost_stress_contract=fixed_target_hold_recovery_events_repriced_at_each_stage_net_nav），没有按 25bp 净值重新决定风控路径。本卡两档费用各自重算状态。两种费用口径必须区分。

年度差异见 regime_metrics.csv。旧单模型与三 seed 重训基线的差异不能全部归因于分钟数据；core_path 必须同时与 base 和 availability_control 对照。
分钟单层的历史改进需要经过完整控制器重新验证。两档费用下候选与对照的排序不同，不能据此重新选模。控制器成本敏感性见 cost_path_differences.csv，净值回撤与风险状态核对见 controller_state_diagnostics.csv。

历史执行价格统一为 NAS 真实分钟及已公布复权事件构建的开盘/首小时结束标记。开盘及 10:30 close 是历史记账标记，不声明真实 09:31/10:31 成交、首次数据接收时点或现金分红资格。模型分钟输入仍为此前冻结的 raw NONE，未用本次复权系数重算特征。
诊断门：{"net_return_above_soxx_both_costs": false, "observed_drawdown_within_30pct_both_costs": false, "return_above_refit_and_availability_controls_both_costs": false, "original_minute_development_gate_pass": false, "independent_forward_gate_pass": false}。
晋升：not_promoted_full_stage1_comparison_diagnostic；原分钟开发门已失败，独立合格前向为 0，objective_achieved=false。比较修正不能重新挑选候选或把复用历史变成新的样本外证据。
下一步：结束此固定对照；保留失败路线。模型优化需要新的有效信息或预先冻结的独立前向对照，不能靠增加同一历史上的参数搜索。
Recorder：9d671fcfb1aa467bb8a9fd83971fb015。行情、大账户及模型引用留 NAS；Git 保留 notebook、瘦卡和代码。
